In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0


Assignment 1. I tried to do the assignment with R since I am familiar with R and used it for all my other assignments in the past. However, after many tries, this IMDB Neural Network Tuning (Keras/TensorFlow) assginment was really hard to do on the R because of the errors that I was not able to fix. So, I am learned how to run Python and did the assignment on Python.  I start with a baseline sentiment model for IMDB reviews and then run a series of controlled experiments (depth, width, loss, activation, regularization). For each setting, I compare validation performance and final test accuracy, then summarize the results in a table and plot.

In [2]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


In [3]:
NUM_WORDS = 10_000

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=NUM_WORDS)

def vectorize_sequences(sequences, dimension=NUM_WORDS):
    results = np.zeros((len(sequences), dimension), dtype=np.float32)
    for i, seq in enumerate(sequences):
        results[i, seq] = 1.0
    return results

x_train = vectorize_sequences(x_train)
x_test  = vectorize_sequences(x_test)

y_train = np.asarray(y_train).astype("float32")
y_test  = np.asarray(y_test).astype("float32")

# Validation split
x_val = x_train[:10_000]
partial_x_train = x_train[10_000:]
y_val = y_train[:10_000]
partial_y_train = y_train[10_000:]

print("Train:", partial_x_train.shape, "Val:", x_val.shape, "Test:", x_test.shape)

c:\Users\hyim\Documents\AML_Assignment1\.venv\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


Train: (15000, 10000) Val: (10000, 10000) Test: (25000, 10000)


In [7]:
def build_model(hidden_layers=2, units=16, activation="relu",
                loss="binary_crossentropy", dropout=0.0, l2=0.0):
    model = keras.Sequential()
    
    reg = keras.regularizers.l2(l2) if l2 > 0 else None
    
    # first hidden layer needs input shape
    model.add(layers.Dense(units, activation=activation, input_shape=(NUM_WORDS,),
                           kernel_regularizer=reg))
    if dropout > 0:
        model.add(layers.Dropout(dropout))
    
    # additional hidden layers
    for _ in range(hidden_layers - 1):
        model.add(layers.Dense(units, activation=activation, kernel_regularizer=reg))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    
    # output layer
    model.add(layers.Dense(1, activation="sigmoid"))
    
    model.compile(optimizer="rmsprop", loss=loss, metrics=["accuracy"])
    return model


In [8]:
def run_experiment(cfg, epochs=20, batch_size=512, patience=2, verbose=0):
    model = build_model(**cfg)
    
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        )
    ]
    
    history = model.fit(
        partial_x_train, partial_y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(x_val, y_val),
        callbacks=callbacks,
        verbose=verbose
    )
    
    val_acc = history.history["val_accuracy"]
    best_val_acc = float(np.max(val_acc))
    best_epoch = int(np.argmax(val_acc) + 1)
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    
    return {
        **cfg,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "test_acc": float(test_acc)
    }


In [9]:
experiments = [
    # Baseline (2 hidden layers)
    {"hidden_layers": 2, "units": 16, "activation": "relu", "loss": "binary_crossentropy", "dropout": 0.0, "l2": 0.0, "model_id": "baseline_L2_U16_relu_bce"},
    
    # 1 vs 3 hidden layers
    {"hidden_layers": 1, "units": 16, "activation": "relu", "loss": "binary_crossentropy", "dropout": 0.0, "l2": 0.0, "model_id": "depth_L1_U16_relu_bce"},
    {"hidden_layers": 3, "units": 16, "activation": "relu", "loss": "binary_crossentropy", "dropout": 0.0, "l2": 0.0, "model_id": "depth_L3_U16_relu_bce"},
    
    # units 32 vs 64
    {"hidden_layers": 2, "units": 32, "activation": "relu", "loss": "binary_crossentropy", "dropout": 0.0, "l2": 0.0, "model_id": "units_L2_U32_relu_bce"},
    {"hidden_layers": 2, "units": 64, "activation": "relu", "loss": "binary_crossentropy", "dropout": 0.0, "l2": 0.0, "model_id": "units_L2_U64_relu_bce"},
    
    # mse loss
    {"hidden_layers": 2, "units": 16, "activation": "relu", "loss": "mse", "dropout": 0.0, "l2": 0.0, "model_id": "loss_L2_U16_relu_mse"},
    
    # tanh activation
    {"hidden_layers": 2, "units": 16, "activation": "tanh", "loss": "binary_crossentropy", "dropout": 0.0, "l2": 0.0, "model_id": "act_L2_U16_tanh_bce"},
    
    # regularization: dropout
    {"hidden_layers": 2, "units": 64, "activation": "relu", "loss": "binary_crossentropy", "dropout": 0.2, "l2": 0.0, "model_id": "reg_dropout0.2_L2_U64"},
    {"hidden_layers": 2, "units": 64, "activation": "relu", "loss": "binary_crossentropy", "dropout": 0.5, "l2": 0.0, "model_id": "reg_dropout0.5_L2_U64"},
    
    # regularization: L2
    {"hidden_layers": 2, "units": 64, "activation": "relu", "loss": "binary_crossentropy", "dropout": 0.0, "l2": 1e-4, "model_id": "reg_l2_1e-4_L2_U64"},
    {"hidden_layers": 2, "units": 64, "activation": "relu", "loss": "binary_crossentropy", "dropout": 0.0, "l2": 1e-3, "model_id": "reg_l2_1e-3_L2_U64"},
]


In [10]:
results = []
for exp in experiments:
    cfg = exp.copy()
    model_id = cfg.pop("model_id")
    out = run_experiment(cfg, verbose=0)
    out["model_id"] = model_id
    results.append(out)
    print(model_id, "-> best_val_acc:", round(out["best_val_acc"], 4), "test_acc:", round(out["test_acc"], 4))

df = pd.DataFrame(results).sort_values("best_val_acc", ascending=False)
df


c:\Users\hyim\Documents\AML_Assignment1\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


baseline_L2_U16_relu_bce -> best_val_acc: 0.8877 test_acc: 0.8822
depth_L1_U16_relu_bce -> best_val_acc: 0.8894 test_acc: 0.8822
depth_L3_U16_relu_bce -> best_val_acc: 0.8872 test_acc: 0.8806
units_L2_U32_relu_bce -> best_val_acc: 0.8875 test_acc: 0.8826
units_L2_U64_relu_bce -> best_val_acc: 0.8883 test_acc: 0.8804
loss_L2_U16_relu_mse -> best_val_acc: 0.8892 test_acc: 0.8779
act_L2_U16_tanh_bce -> best_val_acc: 0.8902 test_acc: 0.8827
reg_dropout0.2_L2_U64 -> best_val_acc: 0.8892 test_acc: 0.8844
reg_dropout0.5_L2_U64 -> best_val_acc: 0.8893 test_acc: 0.8851
reg_l2_1e-4_L2_U64 -> best_val_acc: 0.8888 test_acc: 0.8819
reg_l2_1e-3_L2_U64 -> best_val_acc: 0.8859 test_acc: 0.8811


,hidden_layers,units,activation,loss,dropout,l2,best_val_acc,best_epoch,test_acc,model_id
6,2,16,tanh,binary_crossentropy,0.0,0.0000,0.8902,3,0.88268,act_L2_U16_tanh_bce
1,1,16,relu,binary_crossentropy,0.0,0.0000,0.8894,4,0.88224,depth_L1_U16_relu_bce
8,2,64,relu,binary_crossentropy,0.5,0.0000,0.8893,5,0.88508,reg_dropout0.5_L2_U64
7,2,64,relu,binary_crossentropy,0.2,0.0000,0.8892,3,0.88444,reg_dropout0.2_L2_U64
5,2,16,relu,mse,0.0,0.0000,0.8892,3,0.87792,loss_L2_U16_relu_mse
9,2,64,relu,binary_crossentropy,0.0,0.0001,0.8888,2,0.88192,reg_l2_1e-4_L2_U64
4,2,64,relu,binary_crossentropy,0.0,0.0000,0.8883,4,0.88036,units_L2_U64_relu_bce
0,2,16,relu,binary_crossentropy,0.0,0.0000,0.8877,4,0.88224,baseline_L2_U16_relu_bce
3,2,32,relu,binary_crossentropy,0.0,0.0000,0.8875,3,0.88264,units_L2_U32_relu_bce
2,3,16,relu,binary_crossentropy,0.0,0.0000,0.8872,3,0.88064,depth_L3_U16_relu_bce


Conclusion: Here is the results and conclusion. The baseline model (two hidden layers, 16 units, ReLU activation, and binary crossentropy loss) achieved a validation accuracy of 0.8877 and a test accuracy of 0.8822. I used this as the reference point for all other experiments. Changing the number of hidden layers showed that one hidden layer slightly improved validation accuracy, while three hidden layers slightly reduced test performance. This suggests that adding more depth did not meaningfully improve results and may have added unnecessary complexity.
Increasing the number of hidden units to 32 or 64 did not significantly improve performance. The larger models achieved similar validation accuracy but did not consistently improve test accuracy, indicating possible overfitting. One interesting observation was that simply increasing model complexity did not necessarily improve performance.
Using MSE instead of binary crossentropy resulted in lower test accuracy, confirming that binary crossentropy is more appropriate for this classification task. Switching from ReLU to tanh slightly improved validation accuracy while maintaining similar test performance.
The most noticeable improvement came from regularization. Adding dropout, especially with a rate of 0.5, produced the highest test accuracy (0.8851), suggesting improved generalization.
I chose the model with two hidden layers as the final model because 64 units, ReLU activation, binary crossentropy loss, and dropout of 0.5 performed best model.